Dataset Link: https://huggingface.co/datasets/PurpleAILAB/new_alpaca-SQLi/viewer/default/train?p=2

# Installing Necessary Dependencies

In [1]:
!pip install trl transformers accelerate git+https://github.com/huggingface/peft.git -Uqqq
!pip install datasets sentence_transformers bitsandbytes einops wandb -Uqqq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but

# Loading Necessary Libraries

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, GenerationConfig
from peft import LoraConfig, get_peft_model, PeftConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer
import warnings
warnings.filterwarnings("ignore")

In [3]:
from huggingface_hub import notebook_login
notebook_login()

# Loading Dataset

In [4]:
# Load the dataset directly from Hugging Face
dataset = load_dataset("Pegasus77/sqli")


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

SQLiV3_modified.json:   0%|          | 0.00/4.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11293 [00:00<?, ? examples/s]

### Reformatting dataset to Instruction Template for Mistral Model

In [5]:
# Preprocess to match the instruction-based format
def preprocess_dataset(example):
    instruction = example["instruction"].strip()
    output_text = example["output"].strip()

    # Format as "<s>[INST] instruction input [/INST] output </s>"
    formatted_text = f"<s>[INST] {instruction} [/INST] {output_text}</s>"
    return {"formatted": formatted_text}

# Apply preprocessing to all rows
processed_dataset = dataset.map(preprocess_dataset, remove_columns=dataset["train"].column_names)

# Convert to DatasetDict format
processed_dataset = processed_dataset.rename_column("formatted", "text")


Map:   0%|          | 0/11293 [00:00<?, ? examples/s]

In [6]:
processed_dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 11293
    })
})

In [ ]:
processed_dataset

Dataset({
    features: ['text'],
    num_rows: 2389
})

## Quantizing the Mistral 7B Model to 4-bit

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_has_fp16_weight=True,
)

# Load Mistral model with quantization
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [8]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNo

### Setting LoRA Configuration for Paramter Efficient Fine Tuning

In [9]:
lora_alpha = 64
lora_dropout = 0.05
lora_rank = 32

# Configure LoRA for parameter-efficient fine-tuning
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "lm_head"],
    bias="none",
    task_type="CAUSAL_LM",
)


per_device_train_batch_size = 1
gradient_accumulation_steps = 4
max_steps = 200  # Fine-tune for 200 steps
learning_rate = 2e-4

training_arguments = TrainingArguments(
    output_dir="./Mistral-SQLI-Finetuned",
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=learning_rate,
    max_steps=max_steps,
    warmup_ratio=0.03,
    fp16=True,
    group_by_length=True,
    push_to_hub=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset["train"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
)


Map:   0%|          | 0/11293 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [10]:
import wandb
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [11]:
import time

model.config.use_cache = False
start = time.time()
trainer.train()
end = time.time()
print(f"Fine-tuning completed in {(end - start) / 60:.2f} minutes.")


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: k213340 (k213340-fast-nuces). Use `wandb login --relogin` to force relogin


Step,Training Loss
10,2.229800
20,1.449900
30,1.242400
40,1.086300
50,1.065400
60,0.933600
70,0.844500
80,0.961400
90,0.987700
100,0.948900


Fine-tuning completed in 15.46 minutes.


It took 14.5 mints to fine tune the model on just 120 steps

In [ ]:
(end-start)/60

9.447623145580291

In [13]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/AtifAli121/Mistral-SQLI-Finetuned/commit/8a578356c86cef589158e43015f97b1c4e8ebcae', commit_message='End of training', commit_description='', oid='8a578356c86cef589158e43015f97b1c4e8ebcae', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AtifAli121/Mistral-SQLI-Finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='AtifAli121/Mistral-SQLI-Finetuned'), pr_revision=None, pr_num=None)

## Inference using Fine Tuned Model

In [17]:
import re

def generate_sql_payload(prompt, model, tokenizer):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.pad_token is None else tokenizer.pad_token

    input_encoding = tokenizer(prompt, return_tensors="pt", padding=True).to(device)

    outputs = model.generate(
        input_ids=input_encoding.input_ids,
        attention_mask=input_encoding.attention_mask,
        generation_config=GenerationConfig(
            max_new_tokens=128,
            temperature=0.5,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.3,
            num_return_sequences=1,
            do_sample=True,
        )
    )

    # Decode and clean up results
    generated_queries = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return "\n\n".join(generated_queries)

# Example usage
sql_prompt = "Please help me generate a SQL injection code with the purpose of bypassing basic authentication by exploiting a vulnerable SQL query that improperly handles user input."
print(generate_sql_payload(sql_prompt, trainer.model, trainer.tokenizer))


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Please help me generate a SQL injection code with the purpose of bypassing basic authentication by exploiting a vulnerable SQL query that improperly handles user input.  -3978'   )    )     )   union all select nullnull-- kqjg
   (  case when   (  4256  =  1027  )   then 1 else 0 end   )    and 'xvzl' like 'xvzl' -- qmwe
from dual where 4256  =  1027 group by concat  (  0x7171706a71  (  select   (  elt  (  4256 


In [ ]:
(end-start)

119.00701117515564

In [18]:
trainer.model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k